# Agent 3 — Customer Insight AI Agent
**Owner:** AI/ML Engineer (Member 3)

**Goal:** Analyze real customer reviews to surface the **top complaints** and
**top praises** for a product/business category, using BERT-based sentiment
analysis plus TF-IDF keyword extraction.

**Dataset used:** `data/amazon_reviews_sample.csv` (Amazon consumer reviews)

**Output:** Top positives, top negatives, and a summary insight — returned by
`agent_customer(category)`.

> Note: this notebook uses HuggingFace `transformers` for BERT sentiment, which
> requires downloading a pretrained model the first time (needs internet access).
> A VADER-based fallback is included so the pipeline still runs offline.


In [7]:
!pip install transformers torch

  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/11.2 MB ? eta -:--:--
   ---- ----------------------------------- 1.3/11.2 MB 6.5 MB/s eta 0:00:02
   ------- -------------------------------- 2.1/11.2 MB 5.4 MB/s eta 0:00:02
   ----------- ---------------------------- 3.1/11.2 MB 5.1 MB/s eta 0:00:02
   --------------- ------------------------ 4.5/11.2 MB 5.1 MB/s eta 0:00:02
   -------------------- ------------------- 5.8/11.2 MB 5.3 MB/s eta 0:00:02
   -----

## 1. Imports & load data

In [14]:
import pandas as pd
import numpy as np
import re,json,warnings
warnings.filterwarnings("ignore")
from sklearn.feature_extraction.text import TfidfVectorizer

DATA_DIR="../../data"

amazon=pd.read_csv(f"{DATA_DIR}/amazon_reviews_sample.csv",low_memory=False)

amazon=amazon.rename(columns={
"name":"product",
"categories":"category",
"reviews.rating":"rating",
"reviews.text":"text"
})

amazon=amazon[["product","category","rating","text"]]
amazon["source"]="amazon"
amazon.head()

,product,category,rating,text,source
0,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...","Electronics,iPad & Tablets,All Tablets,Fire Ta...",5.0,This product so far has not disappointed. My c...,amazon
1,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...","Electronics,iPad & Tablets,All Tablets,Fire Ta...",5.0,great for beginner or experienced person. Boug...,amazon
2,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...","Electronics,iPad & Tablets,All Tablets,Fire Ta...",5.0,Inexpensive tablet for him to use and learn on...,amazon
3,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...","Electronics,iPad & Tablets,All Tablets,Fire Ta...",4.0,I've had my Fire HD 8 two weeks now and I love...,amazon
4,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...","Electronics,iPad & Tablets,All Tablets,Fire Ta...",5.0,I bought this for my grand daughter when she c...,amazon


In [15]:
google=pd.read_csv(f"{DATA_DIR}/googleplaystore_user_reviews.csv")

google=google.rename(columns={
"App":"product",
"Translated_Review":"text"
})

google["category"]="mobile app"
google["rating"]=np.nan
google["source"]="google_play"

google=google[["product","category","rating","text"]]
google.head()

,product,category,rating,text
0,10 Best Foods for You,mobile app,NaN,I like eat delicious food. That's I'm cooking ...
1,10 Best Foods for You,mobile app,NaN,This help eating healthy exercise regular basis
2,10 Best Foods for You,mobile app,NaN,NaN
3,10 Best Foods for You,mobile app,NaN,Works great especially going grocery store
4,10 Best Foods for You,mobile app,NaN,Best idea us


## 2. Merge Datasets

In [16]:
reviews=pd.concat([amazon,google],ignore_index=True)
reviews=reviews.dropna(subset=["text"])

def clean_text(text):
    text=str(text).lower()
    text=re.sub(r"[^a-z0-9\s]"," ",text)
    text=re.sub(r"\s+"," ",text)
    return text.strip()

reviews["clean_text"]=reviews["text"].apply(clean_text)

print(reviews.shape)

(72086, 6)


## 3. Text cleaning

In [17]:
def clean_text(text):
    text=str(text).lower()
    text=re.sub(r"[^a-z0-9\s]"," ",text)
    text=re.sub(r"\s+"," ",text)
    return text.strip()

reviews["clean_text"]=reviews["text"].apply(clean_text)

reviews.head()

,product,category,rating,text,source,clean_text
0,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...","Electronics,iPad & Tablets,All Tablets,Fire Ta...",5.0,This product so far has not disappointed. My c...,amazon,this product so far has not disappointed my ch...
1,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...","Electronics,iPad & Tablets,All Tablets,Fire Ta...",5.0,great for beginner or experienced person. Boug...,amazon,great for beginner or experienced person bough...
2,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...","Electronics,iPad & Tablets,All Tablets,Fire Ta...",5.0,Inexpensive tablet for him to use and learn on...,amazon,inexpensive tablet for him to use and learn on...
3,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...","Electronics,iPad & Tablets,All Tablets,Fire Ta...",4.0,I've had my Fire HD 8 two weeks now and I love...,amazon,i ve had my fire hd 8 two weeks now and i love...
4,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...","Electronics,iPad & Tablets,All Tablets,Fire Ta...",5.0,I bought this for my grand daughter when she c...,amazon,i bought this for my grand daughter when she c...


## 4. Load BERT Model

In [18]:
from transformers import pipeline

bert_model=pipeline(
"sentiment-analysis",
model="distilbert-base-uncased-finetuned-sst-2-english"
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

## 5. Sentiment function


In [19]:
def predict_sentiment(text):

    result=bert_model(
    text[:512]
    )[0]

    return result["label"].lower()

## 6. Apply BERT Sentiment

In [20]:
data=reviews.sample(
min(10000,len(reviews)),
random_state=42
)

data["sentiment"]=data["text"].apply(
predict_sentiment
)

data.head()

,product,category,rating,text,source,clean_text,sentiment
65898,Cut the Rope FULL FREE,mobile app,NaN,I loved game long! I 12 I understand ads. This...,NaN,i loved game long i 12 i understand ads this f...,positive
64054,Cookpad,mobile app,NaN,Tq cookpad .... very helpful ... so much cooki...,NaN,tq cookpad very helpful so much cooking knowledge,positive
70401,Doodle Jump,mobile app,NaN,Make More games like,NaN,make more games like,positive
962,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...","Electronics,iPad & Tablets,All Tablets,Fire Ta...",4.0,Very smooth tablet no lag at all also good bat...,amazon,very smooth tablet no lag at all also good bat...,positive
26515,"Amazon Fire Tv,,,\r\nAmazon Fire Tv,,,","Stereos,Remote Controls,Amazon Echo,Audio Dock...",5.0,This product was everything it was advertised ...,amazon,this product was everything it was advertised ...,negative


## 7. Extract Business Domain From Business Idea


In [21]:
def extract_domain(business_idea):

    mapping={
    "food delivery":["food","delivery","restaurant","order","meal"],
    "mobile app":["app","mobile","software"],
    "electronics":["phone","laptop","device","battery"],
    "health":["health","medical","fitness"]
    }

    idea=business_idea.lower()

    for domain,keywords in mapping.items():

        for word in keywords:

            if word in idea:
                return keywords

    return idea.split()

## 8. TF-IDF Engine

In [26]:
def extract_keywords(texts,top_n=5):

    stop_words=[
    "good","great","love","like",
    "best","nice","really",
    "very","app","use",
    "it","is","to",
    "the","and","with",
    "this"
    ]

    vectorizer=TfidfVectorizer(
    stop_words=stop_words,
    ngram_range=(2,3),
    max_features=5000
    )

    matrix=vectorizer.fit_transform(texts)

    scores=np.asarray(
    matrix.sum(axis=0)
    ).flatten()

    words=vectorizer.get_feature_names_out()

    result=[]

    for i in scores.argsort()[::-1]:
        if len(result)>=top_n:
            break

        word=words[i]

        if len(word.split())>=2:
            result.append(word)

    return result

## 9. agent_customer(category)

In [27]:
def agent_customer(business_idea):

    keywords=extract_domain(
    business_idea
    )

    pattern="|".join(keywords)

    filtered=data[
    data["text"].str.contains(
    pattern,
    case=False,
    na=False
    )
    ]

    if len(filtered)<20:
        filtered=data

    positive=filtered[
    filtered["sentiment"]=="positive"
    ]["clean_text"]

    negative=filtered[
    filtered["sentiment"]=="negative"
    ]["clean_text"]

    positives=extract_keywords(
    positive
    )

    negatives=extract_keywords(
    negative
    )

    return {

    "agent":
    "Customer Insight AI Agent",

    "business_idea":
    business_idea,

    "reviews_analyzed":
    len(filtered),

    "positive":
    positives,

    "negative":
    negatives,
  
     "insight":
        f"Customers appreciate {', '.join(positives[:3])}. "
        f"The main problems reported are {', '.join(negatives[:3])}. "
        "Focus on improving these areas to enhance user experience."

    }

## 10. Test the agent

In [28]:
result=agent_customer(
"AI powered fitness app "
)

print(
json.dumps(
result,
indent=4
)
)

{
    "agent": "Customer Insight AI Agent",
    "business_idea": "AI powered fitness app ",
    "reviews_analyzed": 1811,
    "positive": [
        "for my",
        "you can",
        "fire tv",
        "if you",
        "tablet for"
    ],
    "negative": [
        "please fix",
        "every time",
        "can even",
        "for my",
        "nothing happens"
    ],
    "insight": "Customers appreciate for my, you can, fire tv. The main problems reported are please fix, every time, can even. Focus on improving these areas to enhance user experience."
}


## 7. Next steps
- Move `clean_text`, `bert_sentiment`, `top_tfidf_terms` and `agent_customer()` into
  `agent3_customer/sentiment.py` and `agent3_customer/customer_agent.py`.
- For production, run BERT sentiment over the **full** dataset offline (batch job),
  cache the labelled DataFrame, and have `agent_customer()` just filter + summarize
  the cached results (don't run BERT inference per API request).
- Add SpaCy/NLTK lemmatization before TF-IDF to merge similar terms
  (e.g. "battery" / "batteries").
